In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Llama-70B project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-3.1-70B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,   # recommended on A100/H100
    device_map="auto"
)


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|████████████████████████████████████████████████| 30/30 [03:06<00:00,  6.22s/it]


In [3]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [4]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "MedPAIR_1300_Sentence.csv")
print("Columns in dataset:")
print(df.columns.tolist())

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21']


In [5]:
import os
import re
import ast
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from time import time as timer

# === Prepare Output File ===
output_path = paths.DATA / "Llama70B_annotated_MedPAIR_relevancy.csv"

# Create output columns
output_cols = df.columns.tolist() + ['Llama70B_answer', 'Llama70B_raw_response'] + [f"label_{i+1}" for i in range(21)]
sentence_cols = [f'sentence_{i}' for i in range(1, 22)]

# === Check for Existing Progress ===
if os.path.exists(output_path):
    print(f"📂 Found existing file: {output_path}")
    existing_df = pd.read_csv(output_path)
    processed_rows = len(existing_df)
    print(f"✅ Already processed: {processed_rows} rows")
    
    # Load existing results
    results = existing_df.values.tolist()
    start_idx = processed_rows
    print(f"🔄 Resuming from row {start_idx + 1}")
else:
    print(f"📝 No existing file found. Starting fresh...")
    results = []
    start_idx = 0

# === Process Each Row ===
total_rows = len(df)
remaining_rows = total_rows - start_idx

print(f"\n{'='*60}")
print(f"Total rows: {total_rows}")
print(f"Already processed: {start_idx}")
print(f"Remaining to process: {remaining_rows}")
print(f"{'='*60}\n")

if remaining_rows == 0:
    print("✅ All rows already processed!")
else:
    print(f"Processing rows {start_idx + 1} to {total_rows}...\n")

for idx in range(start_idx, total_rows):
    print(f"\n{'='*60}")
    print(f"Processing Row {idx + 1}/{total_rows} (Progress: {idx - start_idx + 1}/{remaining_rows})")
    print(f"{'='*60}")
    
    try:
        # === Extract Sentences and Options ===
        sentences = [df.loc[idx, col] for col in sentence_cols if pd.notna(df.loc[idx, col])]
        num_sentences = len(sentences)
        
        if num_sentences == 0:
            print(f"⚠️ No sentences found for row {idx}, skipping...")
            # Add empty result to maintain row alignment
            result_row = df.loc[idx].tolist() + [None, "No sentences found"] + [None] * 21
            results.append(result_row)
            continue
        
        formatted_sentences = "\n".join(f"{j+1}. {s.strip()}" for j, s in enumerate(sentences))
        options = df.loc[idx, "question_options_x"]
        
        # === Display Memory Info ===
        if torch.cuda.is_available():
            free_mem = torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0)
            print(f"🖥️ CUDA Free Memory: {free_mem / (1024 ** 2):.2f} MB")
        
        # === Format the Prompt ===
        prompt = f"""You are given a list of sentences from a clinical vignette and a multiple-choice clinical question.

Your task is twofold:
(1) Select the most appropriate answer from the given options.
(2) Label *each* of the {num_sentences} sentences as either [High Relevance], [Low Relevance], or [Irrelevant], based on its contribution to answering the question.

Definitions:
[High Relevance]: Sentences that directly support the correct answer with essential clinical information (e.g., diagnosis, key test results).
[Low Relevance]: Sentences that provide useful context or background but are not critical to answering.
[Irrelevant]: Sentences unrelated to the question or not useful for reasoning.

Question and Options:
{options}

Sentences:
{formatted_sentences}

Please provide your response in the following format:
Answer: [Your selected option letter]
Sentence 1: [High Relevance/Low Relevance/Irrelevant]
Sentence 2: [High Relevance/Low Relevance/Irrelevant]
...
(Continue for all {num_sentences} sentences)"""
        
        # === Generate Response Using Chat Template ===
        messages = [{"role": "user", "content": prompt}]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        print(f"🔍 Input Token Count: {len(model_inputs['input_ids'][0])}")
        
        # === Time the Generation ===
        start_time = timer()
        
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=1024,
                do_sample=False
            )
        
        end_time = timer()
        print(f"⏱️ Generation Time: {end_time - start_time:.2f} seconds")
        
        # === Decode Response ===
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # === Display Raw Response ===
        print(f"\n{'='*60}")
        print("Raw Model Response:")
        print(f"{'='*60}")
        print(response)
        print(f"{'='*60}")
        
        # === Extract Answer ===
        answer_match = re.search(r"Answer:\s*([A-J])", response, re.IGNORECASE)
        extracted_answer = answer_match.group(1).upper() if answer_match else None
        
        # === Extract Sentence Labels ===
        labels = [None] * 21  # Initialize all labels as None
        
        for sent_num in range(1, num_sentences + 1):
            # Look for patterns like "Sentence 1: [High Relevance]" or "1. High Relevance"
            pattern = rf"Sentence\s*{sent_num}:\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
            match = re.search(pattern, response, re.IGNORECASE)
            
            if match:
                labels[sent_num - 1] = match.group(1)
            else:
                # Try alternative pattern
                pattern2 = rf"{sent_num}\.\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
                match2 = re.search(pattern2, response, re.IGNORECASE)
                if match2:
                    labels[sent_num - 1] = match2.group(1)
        
        print(f"\n📊 Extracted Answer: {extracted_answer}")
        print(f"📊 Extracted Labels: {labels[:num_sentences]}")
        
        # === Build Result Row ===
        result_row = df.loc[idx].tolist() + [extracted_answer, response] + labels
        results.append(result_row)
        
        # === Save Incrementally Every 10 Rows ===
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results, columns=output_cols)
            temp_df.to_csv(output_path, index=False)
            print(f"\n💾 Saved progress: {idx + 1} rows completed")
        
    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        import traceback
        traceback.print_exc()
        # Add empty result to maintain row alignment
        result_row = df.loc[idx].tolist() + [None, str(e)] + [None] * 21
        results.append(result_row)
    
    # === Clear CUDA Cache ===
    torch.cuda.empty_cache()

# === Final Save ===
print("\n" + "="*60)
print("Processing Complete!")
print("="*60)

final_df = pd.DataFrame(results, columns=output_cols)
final_df.to_csv(output_path, index=False)
print(f"✅ Final results saved to: {output_path}")
print(f"📊 Total rows processed: {len(results)}")
print(f"📊 New rows added in this run: {len(results) - start_idx}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📂 Found existing file: Llama70B_annotated_MedPAIR_relevancy.csv
✅ Already processed: 1030 rows
🔄 Resuming from row 1031

Total rows: 1300
Already processed: 1030
Remaining to process: 270

Processing rows 1031 to 1300...


Processing Row 1031/1300 (Progress: 1/270)
🖥️ CUDA Free Memory: 55.28 MB
🔍 Input Token Count: 492


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


⏱️ Generation Time: 29.60 seconds

Raw Model Response:
Answer: I.

Sentence 1: Low Relevance
The sentence provides context for the evaluation but does not directly contribute to identifying the strongest predictor of an unfavorable prognosis.

Sentence 2: High Relevance
This sentence mentions early speech problems, which is directly related to the correct answer (speech impairment) and provides essential clinical information.

Sentence 3: Low Relevance
Although this sentence describes behaviors that may be relevant to the diagnosis, it does not directly contribute to identifying the strongest predictor of an unfavorable prognosis.

Sentence 4: Irrelevant
Functional constipation is not directly related to the strongest predictor of an unfavorable prognosis in this context.

Sentence 5: Low Relevance
Growth parameters are generally normal, and head circumference is slightly above average but not significantly abnormal. This information does not directly contribute to identifying the stro

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...


Processing Row 1298/1300 (Progress: 268/270)
🖥️ CUDA Free Memory: 13.27 MB
🔍 Input Token Count: 637


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


⏱️ Generation Time: 35.00 seconds

Raw Model Response:
Answer: C.

Sentence 1: [High Relevance] 
Sentence 2: [High Relevance] 
Sentence 3: [Low Relevance] 
Sentence 4: [Low Relevance] 
Sentence 5: [Irrelevant] 
Sentence 6: [Low Relevance] 
Sentence 7: [Irrelevant] 
Sentence 8: [Irrelevant] 
Sentence 9: [Low Relevance] 
Sentence 10: [High Relevance] 
Sentence 11: [Irrelevant] 
Sentence 12: [Irrelevant]

Rationale: 
The patient's symptoms of intermittent breast pain without a palpable mass or nipple discharge, and the absence of other significant findings, suggest that the pain is likely cyclical mastalgia. The most appropriate next step in management would be to provide reassurance and consider topical therapy, such as diclofenac, to alleviate the pain. 

Sentences 1, 2, and 10 are labeled as [High Relevance] because they provide essential information about the patient's symptoms and physical examination findings that support the diagnosis of cyclical mastalgia. 

Sentences 3, 4, 6, and

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Processing Row 1300/1300 (Progress: 270/270)
🖥️ CUDA Free Memory: 13.27 MB
🔍 Input Token Count: 477
⏱️ Generation Time: 23.44 seconds

Raw Model Response:
Answer: C

Sentence 1: [Low Relevance]
This sentence provides some context about the patient's symptoms but does not directly support the diagnosis.

Sentence 2: [High Relevance]
This sentence is crucial in understanding the patient's symptoms and history of mood swings, which is essential for diagnosing cyclothymic disorder.

Sentence 3: [Low Relevance]
This sentence provides some information about the patient's lifestyle but is not directly relevant to the diagnosis.

Sentence 4: [Irrelevant]
This sentence does not provide any useful information for diagnosing the patient's condition.

Sentence 5: [Low Relevance]
This sentence provides some information about the patient's mental status but does not directly support the diagnosis of cyclothymic disorder.

Cyclothymic disorder is characterized by periods of hypomanic symptoms altern

In [ ]:
## Concordance Rate Calculation

In [6]:
Llama70B_Labels = pd.read_csv(paths.DATA / "Llama70B_annotated_MedPAIR_relevancy.csv")
Llama70B_Labels.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'ID', 'centaur_question',
       'sentence_number', 'answer', 'data_source', 'step1_excerpts',
       'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2',
       'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7',
       'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12',
       'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16',
       'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20',
       'sentence_21', 'Llama70B_answer', 'Llama70B_raw_response', 'label_1',
       'label_2', 'label_3', 'label_4', 'label_5', 'label_6', 'label_7',
       'label_8', 'label_9', 'label_10', 'label_11', 'label_12', 'label_13',
       'label_14', 'label_15', 'label_16', 'label_17', 'label_18', 'label_19',
       'label_20', 'label_21'],
      dtype='object')

In [8]:
import pandas as pd

original_path = paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv"
trainee = pd.read_csv(original_path)
trainee.columns
# physician.loc[:, "q2":"q22"]

# Rename label_1–label_20 → q_1–q_20
rename_dict = {f"label_{i}": f"q{i}" for i in range(1, 21)}
Llama70B_Labels = Llama70B_Labels.rename(columns=rename_dict)

# Verify
Llama70B_Labels.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'ID', 'centaur_question',
       'sentence_number', 'answer', 'data_source', 'step1_excerpts',
       'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2',
       'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7',
       'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12',
       'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16',
       'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20',
       'sentence_21', 'Llama70B_answer', 'Llama70B_raw_response', 'q1', 'q2',
       'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13',
       'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21'],
      dtype='object')

In [10]:
df = Llama70B_Labels.copy()

# -------------------------
# 1. Empty rate calculation
# -------------------------
# Define empty as NaN or empty/whitespace-only string
is_empty = df['Llama70B_answer'].isna() | (df['Llama70B_answer'].astype(str).str.strip() == "")
empty_rate = is_empty.mean()

# -------------------------
# 2. Accuracy calculation
# -------------------------
# Compute accuracy only on non-empty predictions
valid_mask = ~is_empty

# Optional normalization for robust matching
pred = df.loc[valid_mask, 'Llama70B_answer'].astype(str).str.strip().str.lower()
gt = df.loc[valid_mask, 'answer'].astype(str).str.strip().str.lower()

accuracy = (pred == gt).mean()

print(f"Llama70B_answer empty rate: {empty_rate:.4f}")
print(f"Accuracy on non-empty predictions: {accuracy:.4f}")

Llama70B_answer empty rate: 0.0046
Accuracy on non-empty predictions: 0.5425


In [11]:
cols_trainee = [f"q{i}" for i in range(1, 21)]

def transform_trainee(value):
    if pd.isna(value):
        return value
    
    val_str = str(value).strip().upper()
    
    # Check if numeric (including floats stored as strings)
    try:
        float(value)
        return "high relevance"
    except ValueError:
        pass
    
    if val_str == "REMOVED":
        return "low relevance"
    else:
        return value

trainee[cols_trainee] = trainee[cols_trainee].apply(lambda col: col.map(transform_trainee))

In [12]:
cols_Llama70B = [f"q{i}" for i in range(1, 21)]

def transform_Llama70B(value):
    if pd.isna(value):
        return value
    val = str(value).strip().lower()
    if val == "not applicable":
        return ""
    elif val == "not relevant":
        return "low relevance"
    else:
        return value

Llama70B_Labels[cols_Llama70B] = Llama70B_Labels[cols_Llama70B].map(transform_Llama70B)

In [13]:
Llama70B_Labels = Llama70B_Labels.sort_values(by='Origin')
for c in cols_Llama70B:
    Llama70B_Labels[c] = Llama70B_Labels[c].apply(lambda x: x.lower() if isinstance(x, str) else x)
Llama70B_Labels[["Origin"] + cols_Llama70B].head(10)

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20
0,ID0002,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,...,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,low relevance,low relevance,irrelevant,low relevance,low relevance,high relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,low relevance,low relevance,low relevance,high relevance,low relevance,low relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,high relevance,high relevance,high relevance,irrelevant,high relevance,high relevance,low relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,low relevance,low relevance,irrelevant,irrelevant,low relevance,low relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,high relevance,low relevance,low relevance,low relevance,irrelevant,high relevance,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,low relevance,low relevance,high relevance,high relevance,irrelevant,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,low relevance,high relevance,low relevance,high relevance,low relevance,high relevance,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,low relevance,high relevance,low relevance,low relevance,high relevance,high relevance,high relevance,high relevance,high relevance,...,high relevance,irrelevant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
import numpy as np
import pandas as pd

# Columns to compare
cols_compare = [f"q{i}" for i in range(1, 20)]

# Create dictionary for fast lookup: Origin → trainee row
trainee_dict = {trainee.loc[i, "Origin"]: trainee.iloc[i] for i in range(len(trainee))}

def calculate_match(row_t, row_p):
    matches = 0
    total_valid = 0  # denominator: number of non-NaN labels

    for c in cols_compare:
        val_t = row_t[c]
        val_p = row_p[c]

        # Only consider this column if both values are not NaN
        if pd.notna(val_t) and pd.notna(val_p):
            total_valid += 1
            if val_t == val_p:
                matches += 1

    if total_valid == 0:
        return np.nan
    else:
        return f"{matches}/{total_valid}"  # numerator / denominator

# Apply only if Origins match
match_results = []
trainee_sources = []

for i in range(len(Llama70B_Labels)):
    origin_p = Llama70B_Labels.loc[i, "Origin"]
    if origin_p in trainee_dict:
        row_t = trainee_dict[origin_p]
        row_p = Llama70B_Labels.iloc[i]
        match_results.append(calculate_match(row_t, row_p))
        trainee_sources.append(row_t.get("data_source_corr", np.nan))
    else:
        match_results.append(np.nan)
        trainee_sources.append(np.nan)

# Add results to DataFrame
Llama70B_Labels["Match?"] = match_results
Llama70B_Labels["data_source_corr_trainee"] = trainee_sources

# View results
Llama70B_Labels[["Origin", "data_source_corr_trainee", "Match?"] + cols_compare].head(50)

,Origin,data_source_corr_trainee,Match?,q1,q2,q3,q4,q5,q6,q7,...,q10,q11,q12,q13,q14,q15,q16,q17,q18,q19
0,ID0002,jama,9/11,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,...,low relevance,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,medxpert,1/7,low relevance,low relevance,irrelevant,low relevance,low relevance,high relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,medbullets,4/7,low relevance,low relevance,low relevance,high relevance,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,jama,6/8,high relevance,high relevance,high relevance,irrelevant,high relevance,high relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,medxpert,4/7,low relevance,low relevance,irrelevant,irrelevant,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,mmlu,4/7,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,jama,5/10,high relevance,low relevance,low relevance,low relevance,irrelevant,high relevance,low relevance,...,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,jama,4/7,low relevance,low relevance,high relevance,high relevance,irrelevant,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,jama,3/6,low relevance,high relevance,low relevance,high relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,jama,9/12,low relevance,high relevance,low relevance,low relevance,high relevance,high relevance,high relevance,...,irrelevant,high relevance,irrelevant,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
def combine_high_relevance_sentences(row):
    """
    Combines sentences where corresponding q columns have 'high relevance'
    """
    high_relevance_sentences = []
    
    # Iterate through q1 to q20
    for i in range(1, 21):
        q_col = f'q{i}'
        sentence_col = f'sentence_{i}'
        
        # Check if q column contains 'high relevance' (case-insensitive)
        if pd.notna(row.get(q_col)) and pd.notna(row.get(sentence_col)):
            if 'high relevance' in str(row[q_col]).lower():
                high_relevance_sentences.append(str(row[sentence_col]))
    
    # Combine all high relevance sentences with a separator
    if high_relevance_sentences:
        return ' '.join(high_relevance_sentences)
    else:
        return None  # or return '' for empty string

# Apply the function to create the new column
Llama70B_Labels['Llama70B_High_Relevance'] = Llama70B_Labels.apply(combine_high_relevance_sentences, axis=1)

# Optional: Check the results
print(f"Total rows with high relevance sentences: {Llama70B_Labels['Qwen14B_High_Relevance'].notna().sum()}")
print("\nSample of the new column:")
print(Llama70B_Labels[['Llama70B_High_Relevance']].head(20))

# Select columns to save (including Match?)
columns_to_save = list(Llama70B_Labels.columns)
# Ensure 'Match?' is included if it exists
if 'Match?' in Llama70B_Labels.columns:
    print("\n'Match?' column found and will be included in the CSV")
else:
    print("\nWarning: 'Match?' column not found in Qwen14B_Labels")

# Save to CSV with all columns (including Match?)
Llama70B_Labels.to_csv(paths.DATA / "Llama70B_annotated_MedPAIR_relevancy.csv", index=False)

Total rows with high relevance sentences: 1288

Sample of the new column:
                              Llama70B_High_Relevance
0   4. External examination was notable for left u...
1   6. Chest auscultation is unremarkable, but a c...
2   4. The patient has lost 12 pounds recently and...
3   1. A woman in her 30s presented with multiple ...
4   7. The amputated finger was left in the classr...
5   1. A 3-year-old white girl is brought to the o...
6   1. A 60-year-old man with a history of IgA kap...
7   3. A slitlamp examination of the right eye rev...
8   2. Computed tomography (CT) showed a right-sid...
9   2. After strabismus surgery, he was noted to h...
10  2. He presented with a 2-day history of painfu...
11  4. An endometrial biopsy specimen shows regula...
12  3. Physical examination reveals a prominent le...
13  1. A 47-year-old man comes to the physician 12...
14  6. Basic laboratory studies are ordered as see...
15  12. On physical exam, the patient has a flat a...
16  1. T

In [18]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to convert 'Match?' to numeric percentage
def match_to_percent(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x) * 100
    if isinstance(x, str) and "/" in x:
        try:
            num, denom = x.split("/")
            return (float(num) / float(denom)) * 100
        except:
            return np.nan
    return np.nan

# Create a numeric column for Match? in percentage
Llama70B_Labels["Match_percent"] = Llama70B_Labels["Match?"].apply(match_to_percent)

# Group by data_source_corr_trainee
grouped = Llama70B_Labels.groupby("data_source_corr_trainee")["Match_percent"]

# Function to compute mean, SD, and 95% CI
def mean_sd_ci(series):
    series = series.dropna()
    n = len(series)
    if n == 0:
        return pd.Series({"mean": np.nan, "sd": np.nan, "CI_lower": np.nan, "CI_upper": np.nan, "n": 0})
    mean = series.mean()
    sd = series.std()  # standard deviation
    sem = stats.sem(series)  # standard error
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)  # 95% CI
    return pd.Series({"mean": mean, "sd": sd, "CI_lower": mean - ci, "CI_upper": mean + ci, "n": n})

# Apply function
summary = grouped.apply(mean_sd_ci).reset_index()

# Pivot from long to wide format
summary = summary.pivot(index='data_source_corr_trainee', columns='level_1', values='Match_percent').reset_index()

# Create beautiful formatted table
summary["Mean (%)"] = summary["mean"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["SD (%)"] = summary["sd"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["95% CI"] = summary.apply(
    lambda row: f"[{row['CI_lower']:.2f}, {row['CI_upper']:.2f}]" if not pd.isna(row['CI_lower']) else "N/A",
    axis=1
)
summary["N"] = summary["n"].astype(int)

# Select and rename columns for final display
result = summary[["data_source_corr_trainee", "N", "Mean (%)", "SD (%)", "95% CI"]]
result.columns = ["Data Source", "N", "Mean (%)", "SD (%)", "95% CI"]

result

,Data Source,N,Mean (%),SD (%),95% CI
0,jama,576,57.45,16.16,"[56.13, 58.77]"
1,medbullets,207,52.06,18.67,"[49.50, 54.62]"
2,medxpert,317,53.21,22.24,"[50.75, 55.67]"
3,mmlu,193,57.44,21.26,"[54.42, 60.46]"


In [20]:
import numpy as np
from scipy import stats

# Drop NaNs
all_matches = Llama70B_Labels["Match_percent"].dropna()

n = len(all_matches)
if n == 0:
    overall_summary = {"N": 0, "Mean (%)": np.nan, "SD (%)": np.nan, "95% CI": (np.nan, np.nan)}
else:
    mean = all_matches.mean()
    sd = all_matches.std()
    sem = stats.sem(all_matches)
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)
    overall_summary = {
        "N": n,
        "Mean (%)": round(mean, 2),
        "SD (%)": round(sd, 2),
        "95% CI": [round(mean - ci, 2), round(mean + ci, 2)]
    }

overall_summary

{'N': 1293,
 'Mean (%)': np.float64(55.55),
 'SD (%)': np.float64(19.11),
 '95% CI': [np.float64(54.5), np.float64(56.59)]}